# MindStride — EEG Motor Imagery Classification

## Cross-subject EEGNet with subject-based split, CV & grid search

In [1]:
!pip uninstall torch torchvision torchaudio -y
!pip install -U --pre torch torchvision torchaudio --index-url https://download.pytorch.org/whl/nightly/cu128

Found existing installation: torch 2.5.0+cu121
Uninstalling torch-2.5.0+cu121:
  Successfully uninstalled torch-2.5.0+cu121
Found existing installation: torchvision 0.20.0+cu121
Uninstalling torchvision-0.20.0+cu121:
  Successfully uninstalled torchvision-0.20.0+cu121
Found existing installation: torchaudio 2.5.0+cu121
Uninstalling torchaudio-2.5.0+cu121:
  Successfully uninstalled torchaudio-2.5.0+cu121
Looking in indexes: https://download.pytorch.org/whl/nightly/cu128
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 657.9/657.9 MB 11.5 MB/s eta 0:00:0000:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 287.2/287.2 MB 11.4 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.8/289.8 MB 11.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.1/139.1 MB 11.5 MB/s eta 0:00:0000:0100:01
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 11.4 MB/s eta 0:00:0000:0100:02
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 12.0 MB/s e

| # | What | Status |
|---|------|--------|
| | **DONE** | |
| 1 | PhysioNet MI data loading (R04, R08, R12) | ✅ |
| 2 | EDA on single subject (PSD, raw signal) | ✅ |
| 3 | Preprocessing: bandpass 7-30 Hz, epoching 0-4s | ✅ |
| 4 | Per-subject z-score normalization | ✅ |
| 5 | Class balancing (downsampling to smallest class) | ✅ |
| 6 | Weighted CrossEntropyLoss everywhere | ✅ |
| 7 | Subject-based split (70/15/15, no leakage) | ✅ |
| 8 | EEGNet — 64 channels, 3 classes | ✅ |
| 9 | EEGNet — 21 motor cortex channels, 3 classes | ✅ |
| 10 | EEGNet — 21ch binary L/R (no rest) | ✅ |
| 11 | Subject-based K-Fold CV (GroupKFold) | ✅ |
| 12 | EEGNet hyperparameter grid search (lr, dropout, f1, d) | ✅ |
| 13 | Final model retrain with best params | ✅ |
| 14 | CSP One-vs-Rest | ✅ |
| 15 | CSP Pairwise (MultiClassCSP) | ✅ |
| 16 | CSP binary (native, for L/R) | ✅ |
| 17 | Grid search over 7 classical ML models (LDA, SVM, RF, KNN, GB, LR, MLP) | ✅ |
| 18 | class_weight='balanced' in ML models | ✅ |
| 19 | Two-stage pipeline: mu-wave gating + binary L/R | ✅ |
| 20 | Comparison of all approaches (bar charts, confusion matrices) | ✅ |
| 21 | Full pipeline repeated on both 64ch and 21ch | ✅ |
| 22 | Preprocessing grid search (tmin/tmax/bandpass/baseline) | ✅ |
| 23 | Joint grid search — preprocessing × all models (EEGNet + 7 ML) | ✅ |
| 24 | Best combo: 4-40 Hz, 0-4s, EEGNet(f1=16,d=2,do=0.25) → **80% test acc** | ✅ |
| | **TODO** | |
| 25 | FBCSP (Filter Bank CSP) | ❌ |
| 26 | Data augmentation (sliding window, noise, warping) | ❌ |
| 27 | Ensemble (voting/stacking of best models) | ❌ |
| 28 | Feature extraction: Hjorth parameters, kurtosis, variance | ❌ |
| 29 | Feature extraction: band powers, spectral entropy, mu/beta ratio | ❌ |
| 30 | Feature extraction: wavelets/STFT → 2D for CNN | ❌ |
| 31 | Feature extraction: connectivity (PLV, coherence) | ❌ |
| 32 | Feature fusion + selection (mutual information, RFE) | ❌ |
| 33 | Subject-adaptive bandpass | ❌ |
| 34 | Riemannian geometry (pyriemann) | ❌ |
| 35 | Attention-based EEGNet | ❌ |
| 36 | Transfer learning (pretrain → fine-tune per subject) | ❌ |
| 37 | Sliding window inference | ❌ |
| 38 | EEGConformer / ATCNet (braindecode) | ❌ |
| 39 | GAN — synthetic EEG epoch augmentation (WGAN-GP / conditional GAN) | ❌ |
| 40 | Variational Autoencoder — latent space + classification on embeddings | ❌ |
| 41 | Contrastive encoder (SimCLR/BYOL-style) — self-supervised pre-training | ❌ |
| 42 | Autoencoder denoising — pre-train encoder → fine-tune classifier | ❌ |
| 43 | Per-trial normalization (z-score per epoch instead of per subject) | ❌ |
| 44 | Euclidean Alignment (covariance matrix centering per subject) | ❌ |
| 45 | Min-max normalization comparison | ❌ |
| 46 | Robust scaling (median/IQR — resistant to EEG artifacts) | ❌ |
| 47 | Normalization strategy grid search (z-score vs min-max vs robust vs EA) | ❌ |
| 48 | Full joint grid search on 64 channels (preprocessing × all models) | ❌ |
| 49 | 64ch vs 21ch comparison (best combos head-to-head, same preprocessing) | ❌ |
| 50 | Statistical analysis (p-values between approaches) | ❌ |
| 51 | Generative models approach | ❌ |
| 52 | Blink trigger (real-time app) | ❌ |

24 out of 51 done — 47%!

In [2]:
!pip install mne kagglehub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 11.6 MB/s eta 0:00:00a 0:00:01


In [3]:
import kagglehub
import numpy as np
import mne
import re
import warnings
from pathlib import Path
from collections import defaultdict

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import StratifiedGroupKFold, cross_val_score
from mne.decoding import CSP
from scipy.signal import butter, sosfilt
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from torch.utils.data import Dataset
from torch.utils.data import DataLoader

# Wyciszenie ostrzeżeń dla czystości logów
warnings.filterwarnings('ignore')
mne.set_log_level('WARNING')

In [4]:
import sys
print(f"Python: {sys.version}")
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")


Python: 3.11.10 | packaged by conda-forge | (main, Oct 16 2024, 01:27:36) [GCC 13.3.0]
PyTorch: 2.12.0.dev20260316+cu128
CUDA available: True


In [5]:
import os
import re
import kagglehub
from pprint import pprint


path = kagglehub.dataset_download("brianleung2020/eeg-motor-movementimagery-dataset")
print("Path to dataset files:", path)

# starts with S + 3 digits; then anything; must end with .edf
pat = re.compile(r"^S\d{3}.*\.edf$", re.IGNORECASE)

subjects_data = {}
other_files = []
desired_runs = ["R04", "R08", "R12"]

for dirname, _, filenames in os.walk(os.path.join(path, "files")):
    for filename in filenames:
        if pat.match(filename) and filename[4:-4] in desired_runs:
            subject = filename[1:4]
            subjects_data.setdefault(subject, []).append(os.path.join(dirname, filename))
        else:
            other_files.append(os.path.join(dirname, filename))

print(f"Found {len(subjects_data)} subjects")
pprint(subjects_data['001'])

100%|██████████| 1.87G/1.87G [02:59<00:00, 11.2MB/s]

Extracting files...


Path to dataset files: /home/jovyan/.cache/kagglehub/datasets/brianleung2020/eeg-motor-movementimagery-dataset/versions/1
Found 109 subjects
['/home/jovyan/.cache/kagglehub/datasets/brianleung2020/eeg-motor-movementimagery-dataset/versions/1/files/S001/S001R04.edf',
 '/home/jovyan/.cache/kagglehub/datasets/brianleung2020/eeg-motor-movementimagery-dataset/versions/1/files/S001/S001R08.edf',
 '/home/jovyan/.cache/kagglehub/datasets/brianleung2020/eeg-motor-movementimagery-dataset/versions/1/files/S001/S001R12.edf']


In [6]:
from tqdm import tqdm  

raw_data = {}
for subject in tqdm(subjects_data, desc="Subjects"):
    raws = []
    for f in tqdm(subjects_data[subject], desc=f"{subject} files", leave=False):
        raw = mne.io.read_raw_edf(f, preload=True)
        if raw.info['sfreq'] == 160.0:
            raws.append(raw)
    if len(raws) == 0:
        print(f"⚠️ Subject {subject}: no valid files, skipping")
        continue
    elif len(raws) == 1:
        raw_data[subject] = raws[0]
    else:
        raw_data[subject] = mne.io.concatenate_raws(raws)

print(f"\nLoaded {len(raw_data)} subjects successfully")

047 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
016 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
017 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
084 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
104 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
029 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
042 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
056 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
095 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
097 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
106 files:   0%|    

⚠️ Subject 088: no valid files, skipping



081 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
010 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
065 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
007 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
075 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
066 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
028 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
015 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
014 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
093 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
027 files:   0%|   

⚠️ Subject 100: no valid files, skipping



038 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
054 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
048 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
005 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
089 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
062 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
101 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
023 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
090 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
063 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
098 files:   0%|   

⚠️ Subject 092: no valid files, skipping



074 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
022 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
011 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
040 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
046 files:   0%|          | 0/3 [00:00<?, ?it/s]
                                                
Subjects: 100%|██████████| 109/109 [00:04<00:00, 24.25it/s]


Loaded 106 subjects successfully


In [7]:
motor_channels = [
    'Fc5.', 'Fc3.', 'Fc1.', 'Fcz.', 'Fc2.', 'Fc4.', 'Fc6.',
    'C5..', 'C3..', 'C1..', 'Cz..', 'C2..', 'C4..', 'C6..',
    'Cp5.', 'Cp3.', 'Cp1.', 'Cpz.', 'Cp2.', 'Cp4.', 'Cp6.',
]

# Only left and right — no rest
event_id_binary = {'left_hand': 2, 'right_hand': 3}

all_X = []
all_y = []
all_subjects = []
skipped = []

for subject in tqdm(raw_data, desc="Epoching subjects (L/R only)"):
    try:
        raw = raw_data[subject].copy()
        # raw.pick(motor_channels)
        raw.pick()
        raw.filter(7., 30., fir_design='firwin', skip_by_annotation='edge', verbose=False)

        events, _ = mne.events_from_annotations(raw, event_id='auto', verbose=False)

        epochs = mne.Epochs(raw, events, event_id_binary, tmin=0.0, tmax=4.0,
                            baseline=None, preload=True, verbose=False)

        X = epochs.get_data().astype(np.float32)  # (n_epochs, 21, 641)
        y = epochs.events[:, -1]                   # 2=left, 3=right
        y = y - 2                                  # remap: 0=left, 1=right

        # Per-subject normalization
        for ch in range(X.shape[1]):
            mean = X[:, ch, :].mean()
            std = X[:, ch, :].std()
            if std > 0:
                X[:, ch, :] = (X[:, ch, :] - mean) / std

        all_X.append(X)
        all_y.append(y)
        all_subjects.append(np.full(len(y), int(subject)))

    except Exception as e:
        skipped.append(subject)
        print(f"⚠️ Subject {subject} failed: {e}")

X_all = np.concatenate(all_X, axis=0)
y_all = np.concatenate(all_y, axis=0)
subjects_all = np.concatenate(all_subjects, axis=0)

# Balance classes
print(f"Before balancing: {np.bincount(y_all)}")
min_count = min(np.bincount(y_all))
balanced_idx = []
for cls in range(2):
    cls_idx = np.where(y_all == cls)[0]
    chosen = np.random.choice(cls_idx, size=min_count, replace=False)
    balanced_idx.append(chosen)
balanced_idx = np.concatenate(balanced_idx)
np.random.shuffle(balanced_idx)

X_all = X_all[balanced_idx]
y_all = y_all[balanced_idx]
subjects_all = subjects_all[balanced_idx]

print(f"\nTotal epochs: {X_all.shape[0]}")
print(f"Shape: {X_all.shape}")
print(f"Class distribution (after balancing): {np.bincount(y_all)}  [0=left, 1=right]")
print(f"Unique subjects: {len(np.unique(subjects_all))}")
if skipped:
    print(f"Skipped subjects: {skipped}")


Epoching subjects (L/R only):  24%|██▎       | 25/106 [00:00<00:00, 211.64it/s]

⚠️ Subject 107 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 047 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 016 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 060 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 017 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 084 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 012 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 104 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 029 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 059 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 042 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 056 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 064 failed: pick() missing 1 required positional argu

Epoching subjects (L/R only): 100%|██████████| 106/106 [00:00<00:00, 433.68it/s]

⚠️ Subject 061 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 101 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 023 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 013 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 090 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 063 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 035 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 098 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 109 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 099 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 082 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 019 failed: pick() missing 1 required positional argument: 'picks'
⚠️ Subject 087 failed: pick() missing 1 required positional argu

ValueError: need at least one array to concatenate

In [ ]:
np.unique(y_all)

In [ ]:
unique_subjects = np.unique(subjects_all)
np.random.shuffle(unique_subjects)

n = len(unique_subjects)
n_train = int(n * 0.7)
n_val = int(n * 0.15)

train_subjects = set(unique_subjects[:n_train])
val_subjects   = set(unique_subjects[n_train:n_train + n_val])
test_subjects  = set(unique_subjects[n_train + n_val:])

train_mask = np.isin(subjects_all, list(train_subjects))
val_mask   = np.isin(subjects_all, list(val_subjects))
test_mask  = np.isin(subjects_all, list(test_subjects))

X_train, y_train = X_all[train_mask], y_all[train_mask]
X_val,   y_val   = X_all[val_mask],   y_all[val_mask]
X_test,  y_test  = X_all[test_mask],  y_all[test_mask]

print(f"Train: {len(X_train)} epochs from {len(train_subjects)} subjects")
print(f"Val:   {len(X_val)} epochs from {len(val_subjects)} subjects")
print(f"Test:  {len(X_test)} epochs from {len(test_subjects)} subjects")
print(f"\nTrain class dist: {np.bincount(y_train)}")
print(f"Val class dist:   {np.bincount(y_val)}")
print(f"Test class dist:  {np.bincount(y_test)}")

In [ ]:
class EEGDataset(Dataset):
    def __init__(self, X: np.ndarray, y: np.ndarray, augment=False):
        """
        X: numpy array (n_samples, n_channels, n_timepoints)
        y: numpy array (n_samples,)
        """
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.long)
        self.augment = augment

    def __len__(self):
        return len(self.y)

    def __getitem__(self, idx):
        x = self.X[idx]
        y = self.y[idx]
        if self.augment:
            # gaussian noise
            x += np.random.normal(0, 0.1, x.shape).astype(np.float32)
            # timeshift by randdom +- 20 samples
            shift = np.random.randint(-20, 21)
            x = np.roll(x, shift, axis=-1)
        return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.long)

In [ ]:
BATCH_SIZE = 64

train_dataset = EEGDataset(X_train, y_train)
val_dataset   = EEGDataset(X_val, y_val)
test_dataset  = EEGDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_dataset,  batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

# FBCSP model

In [ ]:
def eeg_bandpass(eeg, min_freq, max_freq):
    sos_lp = butter(10, max_freq, 'lp', fs=200, output='sos')
    sos_hp = butter(10, min_freq, 'hp', fs=200, output='sos')
    
    eeg_low = sosfilt(sos_lp, eeg)
    eeg_high = sosfilt(sos_hp, eeg_low)
    
    return eeg_high

def eeg2band(eeg: np.ndarray, band='alpha'):
    bands = {'alpha':(8, 12), 'beta':(12,30), 'gamma':(35,99), 'delta':(0.5,4), 'theta':(4,8)}
    band_range = bands[band]
    min_freq, max_freq = band_range[0], band_range[1]
    
    return eeg_bandpass(eeg, min_freq, max_freq)

In [ ]:
alpha = eeg2band(X_train, 'alpha')

In [ ]:
import matplotlib.pyplot as plt

# Wybieramy: [epoka_0, kanał_0, wszystkie_punkty_czasowe]
plt.plot(alpha[0, 0, :])
plt.title("Sygnał alpha: Epoka 0, Kanał 0")
plt.xlabel("Czas (próbki)")
plt.ylabel("Amplituda")
plt.show()

In [ ]:
bands = {'alpha':(8, 12), 'beta':(12,30), 'gamma':(35,99), 'delta':(0.5,4), 'theta':(4,8)}

for b in bands.keys():
    band = eeg2band(X_train, b)
    plt.plot(band[0, 0, :])
    plt.title("Sygnał alpha: Epoka 0, Kanał 0")
    plt.xlabel("Czas (próbki)")
    plt.ylabel("Amplituda")
    plt.show()
